# Notebook 03: Machine Learning Model Benchmarking & Explainable AI (SHAP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/03_Machine_Learning_Crop_Yield_Prediction_Colab.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Dataset:** UN FAOSTAT & Multi-Source Weather Indicators  
**Cloud ML Execution:** Google Colab  

---
## 1. Machine Learning Modeling Framework
In this notebook, we benchmark multiple machine learning regression models (Ridge Regression, Random Forest, XGBoost, LightGBM, CatBoost) to predict crop yields and climate yield anomalies, followed by TreeSHAP interpretability.


In [ ]:
# Step 1: Install Cloud Libraries in Google Colab
!pip install -q xgboost catboost lightgbm shap

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import shap

print('Colab ML environment configured.')


In [ ]:
# Step 2: Load Integrated Dataset
INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW_GITHUB}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'

df = pd.read_csv(INTEGRATED_LOCAL) if os.path.exists(INTEGRATED_LOCAL) else pd.read_csv(INTEGRATED_URL)
print(f'Loaded dataset: {len(df)} rows.')
display(df.head(4))


In [ ]:
# Step 3: Feature Matrix Selection & Train/Test Split
feature_cols = ['tmax', 'tmin', 'temp_mean', 'rain', 'spi', 'heat_days', 'humidity', 'sunshine', 'annual_gdd']
target_col = 'yield_kg_ha'

X = df[feature_cols]
y = df[target_col]

# Temporal Train/Test Split: Train on 2009-2021 (13 years), Test on 2022-2024 (3 years)
train_mask = df['year'] <= 2021
test_mask = df['year'] > 2021

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Training Samples: {len(X_train)} | Test Samples: {len(X_test)}')


In [ ]:
# Step 4: Model Benchmarking & Performance Evaluation
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42, verbose=-1),
    'CatBoost': cb.CatBoostRegressor(iterations=120, learning_rate=0.05, depth=3, random_seed=42, verbose=0)
}

benchmark = []
test_predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_predictions[name] = y_pred
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    benchmark.append({
        'Model': name,
        'R2 Score': round(r2, 4),
        'RMSE (kg/ha)': round(rmse, 2),
        'MAE (kg/ha)': round(mae, 2),
        'MAPE (%)': round(mape, 2)
    })

bench_df = pd.DataFrame(benchmark).sort_values(by='R2 Score', ascending=False)
print('=== MODEL BENCHMARK RESULTS (TEST EVALUATION 2022-2024) ===')
display(bench_df)


In [ ]:
# Step 5: Visualizing Predictions vs Actual FAOSTAT Yield
plt.figure(figsize=(10, 6))
plt.plot(df['year'], df['yield_kg_ha'], 'k-o', lw=2, label='Observed FAOSTAT Yield')

test_years = df.loc[test_mask, 'year']
colors = {'XGBoost': 'blue', 'Random Forest': 'green', 'CatBoost': 'orange'}
for m_name in ['XGBoost', 'Random Forest', 'CatBoost']:
    plt.plot(test_years, test_predictions[m_name], '--s', lw=2, color=colors[m_name], label=f'{m_name} Forecast')

plt.axvspan(2021.5, 2024.5, color='gray', alpha=0.15, label='Blind Test Horizon')
plt.title('Out-of-Sample FAOSTAT Rice Yield Forecasts Across Models', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Yield (kg/ha)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Step 6: Explainable AI with SHAP Attribution
best_xgb = models['XGBoost']
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Global Feature Importance Ranking via Mean |SHAP|', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
